In [ ]:
import pandas as pd
import re

meta = pd.read_csv("../data/processed/jobs_cleaned_20260808.csv")
print(meta.shape)
print(meta["title"].head(30).to_string(index=False))

In [ ]:
JOB_FAMILY = [
    ("excluded",       r"\b(?:teacher|tutor|lecturer)\b|\bno\s+experience\b"),
    ("ml_engineer",    r"\b(?:machine learning|ml)\s+(?:engineer|ops)|\bmlops\b"),
    ("ai_engineer",    r"\bai\s+(?:engineer|developer)"),
    ("software_eng",   r"\bsoftware\s+(?:engineer|developer)|\bbackend\b|\bfull[\s-]?stack\b"),
    ("data_engineer",  r"\bdata\s+engineer|\banalytics\s+engineer"),
    ("data_scientist", r"\bdata\s+scientist|\bdata\s+science\b|\bresearch\s+scientist"),
    ("data_analyst",   r"\bdata\s+analyst|\bdata\s+analytics\b"),
    ("other_analyst",  r"\banalyst\b"),
    ("other_engineer", r"\bengineer\b"),
]

def get_family(title):
    for label, pat in JOB_FAMILY:
        if re.search(pat, title):
            return label
    return "other"

meta["family"] = t.apply(get_family)
print(meta["family"].value_counts())

In [ ]:
print(meta[meta["family"] == "excluded"]["title"].head(20).to_string(index=False))

In [ ]:
o = meta[meta["family"] == "other"]["title"].str.lower()
for pat in ["developer", "product manager", "architect", "data manager", "^lead$"]:
    n = o.str.contains(pat, regex=True).sum()
    print(f"{pat:18s} {n:>4d}")

In [ ]:
sal = pd.read_csv("../data/processed/jobs_salary_20260808.csv")
sal["family"] = sal["title"].str.lower().apply(get_family)
sal = sal[sal["family"] != "excluded"]

print(len(sal))
print(sal["family"].value_counts())

In [ ]:
SENIORITY = [
    ("graduate", r"\b(?:graduate|intern|internship|placement|trainee|apprentice)\b"),
    ("junior",   r"\b(?:junior|entry.level|jr\.?)\b"),
    ("senior",   r"\b(?:senior|snr\.?|sr\.?)\b"),
    ("lead",     r"\b(?:lead|principal|staff|head\s+of|director|chief)\b"),
]

def get_seniority(x):
    for label, pat in SENIORITY:
        if re.search(pat, x):
            return label
    return "unspecified"

sal_big["seniority"] = sal_big["title"].str.lower().apply(get_seniority)

pv = sal_big.pivot_table(index="family", columns="seniority",
                         values="mid", aggfunc="median")
cnt = sal_big.pivot_table(index="family", columns="seniority",
                          values="mid", aggfunc="count")
print("中位数:")
print(pv.round(0).to_string())
print()
print("样本数:")
print(cnt.fillna(0).astype(int).to_string())

In [ ]:
%pip install matplotlib

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["font.size"] = 10

from pathlib import Path
Path("../figures").mkdir(exist_ok=True)
print("ok")

In [ ]:
order = ["data_analyst", "other_analyst", "data_scientist", "ml_engineer"]
labels = ["Data\nAnalyst", "Other\nAnalyst", "Data\nScientist", "ML\nEngineer"]

fig, ax = plt.subplots(figsize=(7, 4.5))

data = [sal_big[sal_big["family"] == f]["mid"].values for f in order]
bp = ax.boxplot(data, labels=labels, showfliers=False, widths=0.55,
                medianprops=dict(color="#c0392b", linewidth=2),
                boxprops=dict(color="#34495e"),
                whiskerprops=dict(color="#34495e"),
                capprops=dict(color="#34495e"))

for i, f in enumerate(order, 1):
    n = (sal_big["family"] == f).sum()
    ax.text(i, ax.get_ylim()[0], f"n={n}", ha="center", va="bottom", fontsize=8, color="#7f8c8d")

ax.set_ylabel("Advertised salary, GBP (midpoint of range)")
ax.set_title("Salary by job family — London, employer-stated salaries only")
ax.yaxis.set_major_formatter(lambda x, p: f"£{int(x/1000)}k")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig("../figures/salary_by_family.png", bbox_inches="tight")
plt.close(fig)
print("saved")

In [ ]:
from IPython.display import Image
Image("../figures/salary_by_family.png")

In [ ]:
import numpy as np

levels = ["graduate", "senior"]
level_labels = ["Graduate level", "Senior level"]
colors = ["#5b8db8", "#c0392b"]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
x = np.arange(len(order))
w = 0.36

for i, (lv, lab, c) in enumerate(zip(levels, level_labels, colors)):
    meds, ns = [], []
    for f in order:
        sub = sal_big[(sal_big["family"] == f) & (sal_big["seniority"] == lv)]["mid"]
        meds.append(sub.median() if len(sub) >= 5 else np.nan)
        ns.append(len(sub))
    pos = x + (i - 0.5) * w
    ax.bar(pos, meds, w, label=lab, color=c, alpha=0.85)
    for p, m, n in zip(pos, meds, ns):
        if not np.isnan(m):
            ax.text(p, m + 1500, f"n={n}", ha="center", fontsize=8, color="#555")

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Median advertised salary, GBP")
ax.set_title("Median salary by job family and seniority — London (n shown per bar)")
ax.yaxis.set_major_formatter(lambda v, p: f"£{int(v/1000)}k")
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig("../figures/salary_by_family_and_level.png", bbox_inches="tight")
plt.close(fig)
print("saved")

In [ ]:
Image("../figures/salary_by_family_and_level.png")

In [ ]:
sk = pd.read_csv("../data/processed/skill_by_seniority_20260811.csv", index_col=0)
print(sk.shape)
print(sk.columns.tolist())
print(sk.head(3).to_string())

In [ ]:
g_col, s_col = "graduate(n=41)", "senior(n=41)"

comp = sk[[g_col, s_col, "overall"]].copy()
comp.index = comp.index.str.replace("has_", "", regex=False)
comp = comp[(comp[g_col] >= 0.05) | (comp[s_col] >= 0.05)]
comp["diff"] = comp[s_col] - comp[g_col]
comp = comp.sort_values("diff")

print(len(comp))
print((comp * 100).round(1).to_string())

In [ ]:
top = pd.concat([comp.head(3), comp.tail(12)]).iloc[::-1]

fig, ax = plt.subplots(figsize=(7.5, 6.5))
y = np.arange(len(top))
colors = ["#5b8db8" if d < 0 else "#c0392b" for d in top["diff"]]

ax.barh(y, top["diff"] * 100, color=colors, alpha=0.85)
ax.set_yticks(y)
ax.set_yticklabels(top.index)
ax.axvline(0, color="#333", linewidth=0.8)
ax.set_xlabel("Percentage points, senior minus graduate")
ax.set_title("Skill mentions: senior vs graduate postings (n=41 each)")
ax.grid(axis="x", alpha=0.3)

ax.text(-24, 4.5, "more common\nin graduate roles",
        fontsize=9, color="#5b8db8", va="center")
ax.text(6, 10.5, "more common\nin senior roles",
        fontsize=9, color="#c0392b", va="center")

fig.tight_layout()
fig.savefig("../figures/skills_graduate_vs_senior.png", bbox_inches="tight")
plt.close(fig)
print("saved")

In [ ]:
Image("../figures/skills_graduate_vs_senior.png")

In [ ]:
meta.to_csv("../data/processed/jobs_with_family_20260814.csv", index=False)
stats.to_csv("../data/processed/salary_by_family_20260814.csv")
comp.to_csv("../data/processed/skills_grad_vs_senior_20260814.csv")

from pathlib import Path
for p in sorted(Path("../figures").iterdir()):
    print(p.name)

In [ ]:
spon = pd.read_csv("../data/processed/sponsor_match_20260812.csv")
print(spon.shape)
print(spon.columns.tolist())
print(spon["confidence"].value_counts())

In [ ]:
sal2 = sal.merge(
    spon[["id", "licensed", "confidence", "is_agency"]],
    on="id", how="left"
)
print(sal2.shape)
print(sal2["confidence"].value_counts(dropna=False))

In [ ]:
sal2["mid"] = (sal2["salary_min"] + sal2["salary_max"]) / 2

grp = sal2.groupby("confidence")["mid"].agg(
    n="count", q1=lambda x: x.quantile(.25),
    median="median", q3=lambda x: x.quantile(.75)
).round(0).astype(int)
print(grp)
print()
print(pd.crosstab(sal2["confidence"], sal2["is_agency"]))

In [ ]:
direct = sal2[~sal2["is_agency"]].copy()

g = direct.groupby("confidence")["mid"].agg(
    n="count", q1=lambda x: x.quantile(.25),
    median="median", q3=lambda x: x.quantile(.75)
).round(0).astype(int)
print("仅非中介:")
print(g)
print()
print(pd.crosstab(direct["confidence"], direct["family"]))

In [ ]:
ds_all = sal[sal["family"] == "data_scientist"].copy()
ds_all["mid"] = (ds_all["salary_min"] + ds_all["salary_max"]) / 2
ds_all["seniority"] = ds_all["title"].str.lower().apply(get_seniority)

print("n =", len(ds_all))
print()
print("按 min / mid / max 的中位数:")
print(ds_all[["salary_min", "mid", "salary_max"]].median().round(0).to_string())
print()
print("层级构成:")
print(ds_all["seniority"].value_counts())
print()
print("地区构成:")
print(ds_all["region"].value_counts())

In [ ]:
sal_ldn = sal[sal["region"] == "London"].copy()
sal_ldn["mid"] = (sal_ldn["salary_min"] + sal_ldn["salary_max"]) / 2
sal_ldn["seniority"] = sal_ldn["title"].str.lower().apply(get_seniority)

print("London 真实薪资:", len(sal_ldn), "/", len(sal))
print()
print(sal_ldn["family"].value_counts())
print()
print(pd.crosstab(sal_ldn["family"], sal_ldn["seniority"]))

In [ ]:
BIG = ["data_analyst", "data_scientist", "other_analyst", "ml_engineer"]
chk = sal[sal["family"].isin(BIG)].copy()
chk["is_ldn"] = chk["region"] == "London"

print(chk.groupby("family")["is_ldn"].agg(["sum", "count", "mean"]).round(3))

In [ ]:
for lv in ["graduate", "senior"]:
    print(f"=== {lv} ===")
    sub = sal_ldn[(sal_ldn["family"].isin(BIG)) & (sal_ldn["seniority"] == lv)]
    print(sub.groupby("family")["mid"].agg(
        n="count", median="median").round(0).astype(int).to_string())
    print()

In [ ]:
sal_big = sal_ldn[sal_ldn["family"].isin(BIG)].copy()
print(len(sal_big))
print(sal_big["family"].value_counts())

In [ ]:
stats_ldn = sal_big.groupby("family")["mid"].agg(
    n="count", q1=lambda x: x.quantile(.25),
    median="median", q3=lambda x: x.quantile(.75)
).round(0).astype(int)

stats_ldn.to_csv("../data/processed/salary_by_family_london_20260814.csv")
sal_ldn.to_csv("../data/processed/salary_london_20260814.csv", index=False)
print(stats_ldn)

In [ ]:
jd = pd.read_csv("../data/raw/jd_full_text_20260811.csv")
sk_df = jd.merge(meta, on="id", how="left")
sk_df["seniority"] = sk_df["title"].str.lower().apply(get_seniority)

print("JD 样本:", len(sk_df))
print()
print("地区构成:")
print(sk_df["region"].value_counts())
print()
print("London 占比:", round((sk_df["region"] == "London").mean() * 100, 1), "%")
print()
print("按层级看 London 占比:")
print(sk_df.groupby("seniority")["region"].apply(
    lambda x: round((x == "London").mean() * 100, 1)))

In [ ]:
%pip freeze | grep -iE "pandas|requests|beautifulsoup|matplotlib|numpy|soupsieve"